# 📊 Exploratory Data Analysis — Online Retail II

**Phase 3** of the Customer Intelligence Platform. This notebook explores the raw transactional data before any modeling, answering:

- What does the data look like? (shape, dtypes, missing values)
- How are `Quantity`, `UnitPrice`, `TotalPrice` distributed?
- What are the temporal patterns (monthly / weekly / hourly)?
- Which products / customers / countries drive revenue?
- How much of the data is cancellations / returns?

> **Data source:** `src.data_loader.load_transactions()` — automatically uses the full UCI dataset when present, otherwise falls back to the synthetic sample.

---

## 0. Setup

In [ ]:
import sys
from pathlib import Path

# Make the project's src/ package importable when the notebook is run from
# notebooks/ or from the project root.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_transactions
from src.visualizations import set_plot_style, save_fig, plot_distribution, plot_top_n

set_plot_style()
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("✅ Setup complete — project root:", PROJECT_ROOT)

## 1. Load the data

`load_transactions()` reads the raw file (`.xlsx` if downloaded, otherwise the synthetic `sample_data.csv`) and normalizes it to the canonical schema.

In [ ]:
df = load_transactions()
print("\nShape:", df.shape)
df.head()

## 2. Data quality overview

In [ ]:
df.info()

In [ ]:
df.describe(include="all")

In [ ]:
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({"missing": missing, "%": missing_pct}).query("missing > 0")
print("Missing values:")
missing_df

In [ ]:
if not missing_df.empty:
    fig, ax = plt.subplots(figsize=(9, 4))
    sns.barplot(x=missing_df["%"], y=missing_df.index, ax=ax, palette="rocket")
    ax.set_title("Missing values by column (%)")
    ax.set_xlabel("% missing")
    save_fig(fig, "missing_values")
    plt.show()

## 3. Numeric distributions

Transactional data is typically heavily right-skewed, so we inspect both the raw and `log1p`-transformed views.

In [ ]:
for col in ["Quantity", "UnitPrice", "TotalPrice"]:
    fig = plot_distribution(df, col, log=True, title=f"{col} (log-transformed)")
    save_fig(fig, f"dist_{col.lower()}")
    plt.show()

## 4. Temporal patterns

In [ ]:
ts = df.copy()
ts["InvoiceDate"] = pd.to_datetime(ts["InvoiceDate"], errors="coerce")
ts = ts.dropna(subset=["InvoiceDate"])
ts["Month"] = ts["InvoiceDate"].dt.to_period("M").dt.to_timestamp()
ts["DayOfWeek"] = ts["InvoiceDate"].dt.day_name()
ts["Hour"] = ts["InvoiceDate"].dt.hour

monthly = ts.groupby("Month").agg(transactions=("InvoiceNo", "count"), revenue=("TotalPrice", "sum"))
monthly.head()

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(monthly.index, monthly["transactions"], color="#4C72B0", marker="o", label="transactions")
ax1.set_ylabel("Transactions", color="#4C72B0")
ax2 = ax1.twinx()
ax2.plot(monthly.index, monthly["revenue"], color="#C44E52", marker="s", label="revenue")
ax2.set_ylabel("Revenue (£)", color="#C44E52")
ax1.set_title("Monthly transactions vs revenue")
fig.autofmt_xdate()
save_fig(fig, "monthly_transactions_revenue")
plt.show()

In [ ]:
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
by_day = ts["DayOfWeek"].value_counts().reindex(day_order).fillna(0)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.barplot(x=by_day.values, y=by_day.index, ax=axes[0], palette="crest")
axes[0].set_title("Transactions by day of week")
axes[0].set_xlabel("transactions")

sns.histplot(ts["Hour"], bins=range(6, 21), ax=axes[1], color="#8172B3")
axes[1].set_title("Transactions by hour of day")
axes[1].set_xlabel("hour")
save_fig(fig, "temporal_day_hour")
plt.show()

## 5. Products

Which SKUs move the most volume and generate the most revenue?

In [ ]:
fig = plot_top_n(df, by="Description", n=15, value_col="Quantity", agg="sum",
                 title="Top 15 products by units sold")
save_fig(fig, "top_products_quantity")
plt.show()

In [ ]:
fig = plot_top_n(df, by="Description", n=15, value_col="TotalPrice", agg="sum",
                 title="Top 15 products by revenue", palette="flare")
save_fig(fig, "top_products_revenue")
plt.show()

## 6. Customers

In [ ]:
cust = df.dropna(subset=["CustomerID"])
per_customer = cust.groupby("CustomerID").agg(
    transactions=("InvoiceNo", "nunique"),
    revenue=("TotalPrice", "sum"),
)
print("Customers:", len(per_customer))
per_customer.describe()

In [ ]:
top_customers = per_customer["revenue"].sort_values(ascending=False).head(10)
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(x=top_customers.values, y=top_customers.index.astype(str), ax=ax, palette="mako")
ax.set_title("Top 10 customers by revenue")
ax.set_xlabel("revenue (£)")
ax.set_ylabel("CustomerID")
save_fig(fig, "top_customers_revenue")
plt.show()

## 7. Geography

In [ ]:
country = df.groupby("Country").agg(transactions=("InvoiceNo", "count"), revenue=("TotalPrice", "sum"))
country = country.sort_values("revenue", ascending=False)

# UK dominates, so we compare UK vs the rest separately.
uk_share = df["Country"].eq("United Kingdom").mean() * 100
print(f"United Kingdom share of transactions: {uk_share:.1f}%")
print("\nTop 10 countries by revenue:")
country.head(10)

In [ ]:
top_countries = country.drop(index="United Kingdom", errors="ignore").head(10)
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(x=top_countries["revenue"], y=top_countries.index, ax=ax, palette="viridis")
ax.set_title("Top 10 non-UK countries by revenue")
ax.set_xlabel("revenue (£)")
save_fig(fig, "top_countries_revenue")
plt.show()

## 8. Returns & cancellations

Cancellations are invoices starting with `"C"`; returns are rows with non-positive `Quantity` or `UnitPrice`.

In [ ]:
is_cancelled = df["InvoiceNo"].astype(str).str.startswith("C")
non_positive = (pd.to_numeric(df["Quantity"], errors="coerce") <= 0) | (pd.to_numeric(df["UnitPrice"], errors="coerce") <= 0)
is_return = is_cancelled | non_positive

net_revenue = df["TotalPrice"].sum()
return_value = df.loc[is_return, "TotalPrice"].sum()

print(f"Rows flagged as returns/cancellations : {is_return.sum():,} ({is_return.mean()*100:.2f}%)")
print(f"Gross revenue (all rows)              : £{net_revenue:,.2f}")
print(f"Value removed by returns              : £{return_value:,.2f}")
print(f"Net revenue after removing returns    : £{net_revenue - return_value:,.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
returns_value = pd.to_numeric(df.loc[is_return, "TotalPrice"], errors="coerce").dropna()
sns.histplot(returns_value, bins=50, ax=ax, color="#C44E52")
ax.set_title("Distribution of return/cancellation values")
ax.set_xlabel("TotalPrice (£) — typically negative")
save_fig(fig, "returns_distribution")
plt.show()

## 9. Summary & next steps

### Key takeaways
- The data is **heavily right-skewed** — log transforms and outlier handling are essential before modeling.
- **United Kingdom dominates** (>90% of transactions); segmentation should account for the long tail of countries.
- **Returns/cancellations** are a small fraction of rows but materially affect net revenue — they are dropped by `clean_transactions()` for RFM/LTV/churn.
- There is clear **seasonality** (holiday peak) and a **business-hours** pattern.
- A handful of **products and customers** drive a disproportionate share of revenue — a strong case for segmentation.

### Next notebook
→ **`02_rfm.ipynb`** (Phase 4): build Recency–Frequency–Monetary features and segment customers with K-Means.

---
© 2026 AdamAI-Systems.